# 🏠🏠🏠 Projet Kaggle : Régression : Segmentation 🏠🏠🏠

## Initialisation

### Importation des bibliothèques nécessaires


In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import statsmodels.api as sm
import statsmodels.formula.api as smf
from joblib import Parallel, delayed
from sklearn.metrics import (
    max_error,
    mean_absolute_error,
    median_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import train_test_split

### Importation des données


In [2]:
with open("../data/processed/dtype_dict.json") as f:
    dtype_dict = json.load(f)

train = pd.read_csv(
    "../data/processed/train.csv",
    delimiter=",",
    encoding="utf-8",
    index_col="Id",
    dtype=dtype_dict,
)

test = pd.read_csv(
    "../data/processed/test.csv",
    delimiter=",",
    encoding="utf-8",
    index_col="Id",
    dtype=dtype_dict,
)

dfs = [train, test]

### Reprise des transformations intéressantes


In [3]:
neighborhoods_to_keep = [
    "Brookside",
    "Clear Creek",
    "Crawford",
    "Northridge",
    "Northridge Heights",
    "Stone Brook",
    "Veenker",
]

for df in dfs:
    df["Neighborhood_agg2"] = np.where(
        df["Neighborhood"].isin(neighborhoods_to_keep), df["Neighborhood"], "Autre"
    )
    df.rename(
        columns={"1stFlrSF": "FirstFlrSF", "2ndFlrSF": "SecondFlrSF"}, inplace=True
    )
    df["FullBath_tot"] = df["FullBath"] + df["BsmtFullBath"]
    df["HalfBath_tot"] = df["HalfBath"] + df["BsmtHalfBath"]

Test d'une CAH pour éventuellement challenger la segmentation, mais je ne suis pas fan.

```python
numerical_cols = [
    col for col in train.columns if pd.api.types.is_any_real_numeric_dtype(train[col])
]

categorical_cols = [col for col in train.columns if train[col].dtype == "object"]

scaler = StandardScaler()
quanti_scaled_df = pd.DataFrame(
    scaler.fit_transform(train[numerical_cols]),
    index=train.index,
    columns=numerical_cols,
)

encoder = OneHotEncoder(drop="first", sparse_output=False)
quali_encoded_df = pd.DataFrame(
    encoder.fit_transform(train[categorical_cols]),
    index=train.index,
    columns=encoder.get_feature_names_out(categorical_cols),
)

df_normalized = pd.concat([quanti_scaled_df, quali_encoded_df], axis=1)

dist_quanti = pairwise_distances(quanti_scaled_df, metric="euclidean")
dist_quali = pairwise_distances(quali_encoded_df, metric="hamming")

alpha = 0.5  # Poids des quantitatives
beta = 1 - alpha  # Poids des qualitatives

dist_mixed = alpha * dist_quanti + beta * dist_quali

# Calcul de la matrice de distance
Z = linkage(dist_mixed, method="ward")

fig = plt.figure(figsize=(25, 10))
dn = dendrogram(Z)
plt.show()

# Effectuer la CAH avec différentes coupes du dendrogramme
max_clusters = 10  # Nombre maximal de clusters à considérer
silhouette_scores = []

for num_clusters in range(2, max_clusters + 1):
    clusters = fcluster(Z, num_clusters, criterion="maxclust")
    silhouette_avg = silhouette_score(df_normalized, clusters)
    silhouette_scores.append(silhouette_avg)

# Tracer le graphique du score de silhouette moyen pour chaque nombre de clusters
plt.plot(range(2, max_clusters + 1), silhouette_scores, marker="o")
plt.title("Score de silhouette moyen pour chaque nombre de clusters")
plt.xlabel("Nombre de clusters")
plt.ylabel("Score de silhouette moyen")
plt.show()

# Trouver le nombre optimal de clusters qui maximise le score de silhouette moyen
optimal_num_clusters = (
    np.argmax(silhouette_scores) + 2
)  # +2 car on commence à 2 clusters
print("Nombre optimal de clusters :", optimal_num_clusters)

# Demander à l'utilisateur de choisir le nombre de clusters
num_clusters = 3

# Effectuer la CAH avec le nombre de clusters choisi
clusters = fcluster(Z, num_clusters, criterion="maxclust")

# Ajouter les informations de cluster dans le DataFrame
train["Cluster"] = clusters

# Afficher les statistiques des clusters
cluster_stats = (
    train[
        [
            "TotalBsmtSF",
            "FirstFlrSF",
            "SecondFlrSF",
            "GarageArea",
            "LotFrontage",
            "LotArea",
            "HalfBath_tot",
            "FullBath_tot",
        ]
        + ["Cluster"]
    ]
    .groupby("Cluster")
    .mean()
)
cluster_stats
```


## Reprise de la dernière régression et segmentation

### Création de la formule


In [4]:
selection = [
    "TotalBsmtSF",
    "FirstFlrSF",
    "SecondFlrSF",
    "GarageArea",
    "LotFrontage",
    "LotArea",
    "HalfBath_tot",
    "FullBath_tot",
    "LotConfig_agg",
    "GarageQual_agg",
    "Fireplaces_optb",
    "KitchenQual",
    "BsmtExposure_ord",
    "BsmtQual",
    "ExterQual",
    "Exterior1st_agg",
    "OverallQual_ord",
    "Neighborhood_agg2",
    "MSZoning",
    "BsmtFinType1_ord",
    "CentralAir",
    "HouseAgeAtSale",
]

debut_formule = "SalePrice ~"

debut_cat = " + C("

fin_cat = ")"

formule = debut_formule

for col in selection:
    if pd.api.types.is_any_real_numeric_dtype(train[col]) or col.endswith("_ord"):
        formule = str(formule) + " + " + str(col)
    else:
        formule = str(formule) + debut_cat + str(col) + fin_cat

### Découpage à 650 SF

NB : C'est un compromis volume/forme des résidus.


In [5]:
train_pt1 = train[train["SecondFlrSF"] < 650].copy()
train_pt2 = train[train["SecondFlrSF"] >= 650].copy()

### Séparation en train test

NB : Réduction du rapport train/test, le volume de donnée étant réduit


In [42]:
df_train_pt1, df_test_pt1 = train_test_split(
    train_pt1.drop(columns=["Alley"]),
    test_size=0.2,
    random_state=42,
    stratify=train_pt1["MSZoning"],
)

df_train_pt2, df_test_pt2 = train_test_split(
    train_pt2.drop(columns=["Alley"]),
    test_size=0.2,
    random_state=42,
    stratify=train_pt2["MSZoning"],
)

### Définition des modèles

NB : Les formules vont dans un premier temps être les mêmes.


In [7]:
reg1 = smf.glm(
    formula=formule,
    data=df_train_pt1,
    family=sm.families.Gamma(link=sm.families.links.Identity()),
)

reg2 = smf.glm(
    formula=formule,
    data=df_train_pt2,
    family=sm.families.Gamma(link=sm.families.links.Identity()),
)

c:\Users\guill\miniconda3\envs\house-prices-env\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:308: DomainWarning: The Identity link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "
c:\Users\guill\miniconda3\envs\house-prices-env\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:308: DomainWarning: The Identity link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "


### Entraînement des modèles


In [8]:
res1 = reg1.fit()
res2 = reg2.fit()

### Analyses globales


In [9]:
res1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:              SalePrice   No. Observations:                  800
Model:                            GLM   Df Residuals:                      752
Model Family:                   Gamma   Df Model:                           47
Link Function:               Identity   Scale:                        0.015029
Method:                          IRLS   Log-Likelihood:                -8982.4
Date:                Fri, 28 Feb 2025   Deviance:                       11.473
Time:                        18:04:28   Pearson chi2:                     11.3
No. Iterations:                    18   Pseudo R-squ. (CS):             0.9999
Covariance Type:            nonrobust                                         
===========================================================================================================================
                                                              coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------
Intercept                                                3.299e+04   1.08e+04      3.048      0.002    1.18e+04    5.42e+04
C(LotConfig_agg)[T.Cul-de-sac]                           1.003e+04   3971.282      2.526      0.012    2249.267    1.78e+04
C(LotConfig_agg)[T.Frontage on 2 / 3 sides of property] -3997.2258   3634.051     -1.100      0.271   -1.11e+04    3125.383
C(LotConfig_agg)[T.Inside lot]                          -1728.5452   1652.612     -1.046      0.296   -4967.606    1510.516
C(GarageQual_agg)[T.Good]                                1.354e+04   7041.246      1.923      0.054    -259.730    2.73e+04
C(GarageQual_agg)[T.No garage]                            728.1579   3385.296      0.215      0.830   -5906.899    7363.215
C(GarageQual_agg)[T.Typical/Average]                     4146.9908   2949.037      1.406      0.160   -1633.015    9926.997
C(Fireplaces_optb)[T.[0.50, 1.50)]                       5080.0644   1618.613      3.139      0.002    1907.641    8252.488
C(Fireplaces_optb)[T.[1.50, inf)]                        3827.6736   3471.293      1.103      0.270   -2975.935    1.06e+04
C(KitchenQual)[T.Fair]                                  -1.666e+04   5716.540     -2.914      0.004   -2.79e+04   -5452.709
C(KitchenQual)[T.Good]                                  -1.412e+04   5052.170     -2.796      0.005    -2.4e+04   -4221.401
C(KitchenQual)[T.Typical/Average]                       -2.178e+04   5056.926     -4.307      0.000   -3.17e+04   -1.19e+04
C(BsmtQual)[T.Fair (70-79 inches)]                      -4.061e+04   6565.161     -6.186      0.000   -5.35e+04   -2.77e+04
C(BsmtQual)[T.Good (90-99 inches)]                      -3.054e+04   5503.774     -5.549      0.000   -4.13e+04   -1.98e+04
C(BsmtQual)[T.No basement]                              -1.476e+04   7913.239     -1.865      0.062   -3.03e+04     748.865
C(BsmtQual)[T.Typical (80-89 inches)]                   -3.281e+04   5798.358     -5.659      0.000   -4.42e+04   -2.14e+04
C(ExterQual)[T.Excellent]                                4.102e+04   1.09e+04      3.754      0.000    1.96e+04    6.24e+04
C(ExterQual)[T.Fair]                                     7287.4207   4354.901      1.673      0.094   -1248.029    1.58e+04
C(ExterQual)[T.Good]                                     9203.9151   2837.631      3.244      0.001    3642.261    1.48e+04
C(Exterior1st_agg)[T.Cement Board]                      -1.279e+04   4691.657     -2.726      0.006    -2.2e+04   -3594.575
C(Exterior1st_agg)[T.Metal Siding]                      -8106.4469   3536.215     -2.292      0.022    -1.5e+04   -1175.594
C(Exterior1st_agg)[T.Other Exterior Materials]          -7013.6621   4028.888     -1.741      0.082   -1.49

In [10]:
res2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:              SalePrice   No. Observations:                  367
Model:                            GLM   Df Residuals:                      319
Model Family:                   Gamma   Df Model:                           47
Link Function:               Identity   Scale:                        0.015972
Method:                          IRLS   Log-Likelihood:                -4234.9
Date:                Fri, 28 Feb 2025   Deviance:                       5.5437
Time:                        18:04:29   Pearson chi2:                     5.10
No. Iterations:                    19   Pseudo R-squ. (CS):             0.9990
Covariance Type:            nonrobust                                         
===========================================================================================================================
                                                              coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------
Intercept                                               -2.565e+04   1.89e+04     -1.356      0.175   -6.27e+04    1.14e+04
C(LotConfig_agg)[T.Cul-de-sac]                           5891.2182   7143.106      0.825      0.410   -8109.013    1.99e+04
C(LotConfig_agg)[T.Frontage on 2 / 3 sides of property] -8931.6718   7760.465     -1.151      0.250   -2.41e+04    6278.560
C(LotConfig_agg)[T.Inside lot]                           2220.4251   3490.637      0.636      0.525   -4621.099    9061.949
C(GarageQual_agg)[T.Good]                                4.327e+04   1.65e+04      2.622      0.009    1.09e+04    7.56e+04
C(GarageQual_agg)[T.No garage]                            1.96e+04   9507.086      2.062      0.039     965.820    3.82e+04
C(GarageQual_agg)[T.Typical/Average]                     1.527e+04   6450.588      2.368      0.018    2631.758    2.79e+04
C(Fireplaces_optb)[T.[0.50, 1.50)]                       5612.1126   3247.465      1.728      0.084    -752.801     1.2e+04
C(Fireplaces_optb)[T.[1.50, inf)]                        1.185e+04   5727.498      2.069      0.039     627.105    2.31e+04
C(KitchenQual)[T.Fair]                                  -3.883e+04   1.23e+04     -3.159      0.002   -6.29e+04   -1.47e+04
C(KitchenQual)[T.Good]                                  -3.402e+04   8377.304     -4.061      0.000   -5.04e+04   -1.76e+04
C(KitchenQual)[T.Typical/Average]                       -3.688e+04   8540.277     -4.319      0.000   -5.36e+04   -2.01e+04
C(BsmtQual)[T.Fair (70-79 inches)]                      -1.133e+04   1.27e+04     -0.891      0.373   -3.62e+04    1.36e+04
C(BsmtQual)[T.Good (90-99 inches)]                      -3.445e+04   9116.014     -3.779      0.000   -5.23e+04   -1.66e+04
C(BsmtQual)[T.No basement]                              -4932.2925   1.69e+04     -0.292      0.770    -3.8e+04    2.81e+04
C(BsmtQual)[T.Typical (80-89 inches)]                   -2.946e+04      1e+04     -2.941      0.003   -4.91e+04   -9828.949
C(ExterQual)[T.Excellent]                                1.714e+04   1.66e+04      1.034      0.301   -1.54e+04    4.96e+04
C(ExterQual)[T.Fair]                                    -1.562e+04   1.26e+04     -1.238      0.216   -4.04e+04    9108.189
C(ExterQual)[T.Good]                                     3227.2615   4164.535      0.775      0.438   -4935.077    1.14e+04
C(Exterior1st_agg)[T.Cement Board]                      -1728.1631   1.21e+04     -0.143      0.886   -2.54e+04    2.19e+04
C(Exterior1st_agg)[T.Metal Siding]                      -5350.8734   9222.115     -0.580      0.562   -2.34e+04    1.27e+04
C(Exterior1st_agg)[T.Other Exterior Materials]          -1764.0929   1.07e+04     -0.165      0.869   -2.27

## Analyses des performances

### Définition des différents thèmes et figures plotly


In [11]:
# Template personnalisé
monTheme = go.layout.Template(
    layout=dict(
        template="simple_white",
        autosize=True,
        font=dict(family="Arial", size=15, color="#000000"),
        title=dict(font=dict(size=35, family="Arial"), x=0.5),
        xaxis=dict(tickangle=-35, automargin=True),
        yaxis=dict(tickangle=-35, automargin=True),
    )
)

# Enregistrement du template
pio.templates["monTheme"] = monTheme

# Définition du template comme template par défaut
pio.templates.default = "monTheme"

# Même principe, style de boutons par defaut
# Ne peut pas rentrer dans les templates
styleBoutons = dict(
    bgcolor="#6B6B6B",
    bordercolor="#000000",
    borderwidth=1.5,
    direction="right",
    font_weight=700,
    showactive=True,
    type="buttons",
    x=1,
    xanchor="right",
    y=1.2,
    yanchor="top",
)

mesPolices = {
    "font-size": 25,
    "font-family": "Arial",
    "font-weight": 700,
    "color": "Black",
}

rouge = "rgb(200, 10, 10)"

res = "Résidus"

In [12]:
def plot_perf(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "MAE": [mean_absolute_error(y_true, y_pred)],
            "RMSE": [root_mean_squared_error(y_true, y_pred)],
            "MEE": [median_absolute_error(y_true, y_pred)],
            "ME": [max_error(y_true, y_pred)],
            "R2": [r2_score(y_true, y_pred)],
        }
    )

In [13]:
def residuals_density(df: pd.DataFrame, res_col: str):
    residuals_density = px.histogram(
        df, x=res_col, marginal="box", color_discrete_sequence=[rouge]
    )

    residuals_density.update_layout(
        title_text="Répartition des résidus",
        xaxis_title=res,
        yaxis_title="Nombre",
        showlegend=False,
    )

    residuals_density.show()

In [14]:
def scat_res_price(df: pd.DataFrame, res_col: str, target_col: str):
    scat_res_price = px.scatter(
        df,
        x=target_col,
        y=res_col,
        color_discrete_sequence=[rouge],
    )

    scat_res_price.update_layout(
        title_text="Résidus (valeur réelle - valeur prédite) en fonction du Prix des maisons",
        xaxis_title="Prix de vente",
        yaxis_title=res,
    )

    scat_res_price.show()

In [15]:
def scat(df, first_col: str, res_col: str, numerical_cols: list):
    # Définition de la figure type nuages de points
    scat = go.Figure(
        go.Scatter(
            x=df[first_col],
            y=df[res_col],
            mode="markers",
            marker_color=rouge,
        )
    )

    boutons_x = [
        dict(
            label=f"x - {x}",
            method="update",
            args=[
                {"x": [df[x]]},
                {"xaxis": {"title": x}},
            ],
        )
        for x in numerical_cols
    ]

    # Mise à jour du layout
    scat.update_layout(
        title_text="Relation entre les résidus et la variable quantitative séléctionnée",
        xaxis_title=first_col,
        yaxis_title=res,
        updatemenus=[
            dict(
                buttons=boutons_x,
                direction="up",  # Set to 'down' or 'up' for dropdown
                showactive=True,
                x=1,
                xanchor="right",
                y=-0.25,
                yanchor="bottom",  # Custom styles specified here
                bgcolor=styleBoutons["bgcolor"],
                bordercolor=styleBoutons["bordercolor"],
                borderwidth=styleBoutons["borderwidth"],
            ),
        ],
    )

    # Affichage de la figure
    scat.show()

In [16]:
def violin(df, first_col: str, res_col: str, categorical_cols: list):
    violin = go.Figure(
        go.Violin(
            x=df[first_col],
            y=df[res_col],
            fillcolor=rouge,
            line_color="black",
            marker_color="black",
            box_visible=True,
            meanline_visible=True,
        )
    )

    boutons_y = [
        dict(
            label=f"x - {x}",
            method="update",
            args=[
                {"x": [df[x]]},
                {"xaxis": {"title": x}},
            ],
        )
        for x in categorical_cols
    ]

    # Mise à jour du layout
    violin.update_layout(
        title_text="Relation entre les résidus et la variable qualitative sélectionnée",
        xaxis_title=first_col,
        yaxis_title=res,
        updatemenus=[
            dict(
                buttons=boutons_y,
                direction="up",  # Set to 'down' or 'up' for dropdown
                showactive=True,
                x=1,
                xanchor="right",
                y=-0.25,
                yanchor="bottom",  # Custom styles specified here
                bgcolor=styleBoutons["bgcolor"],
                bordercolor=styleBoutons["bordercolor"],
                borderwidth=styleBoutons["borderwidth"],
            ),
        ],
    )

    # Affichage de la figure
    violin.show()

### Calculs des prédictions


In [17]:
df_test_pt1["SalePrice_pred_stack"] = res1.predict(df_test_pt1)
df_test_pt2["SalePrice_pred_stack"] = res2.predict(df_test_pt2)

df_test_stack = pd.concat([df_test_pt1, df_test_pt2])

### Quelques métriques bien connues


In [18]:
plot_perf(df_test_stack["SalePrice"], df_test_stack["SalePrice_pred_stack"])

,MAE,RMSE,MEE,ME,R2
0,20080.475881,41635.281342,13196.185099,528007.663036,0.753672


### Forme des résidus


In [19]:
df_test_stack["residus"] = (
    df_test_stack["SalePrice"] - df_test_stack["SalePrice_pred_stack"]
)
residuals_density(df_test_stack, "residus")

### Résidus en fonction du Prix de vente


In [20]:
scat_res_price(
    df_test_stack,
    "residus",
    "SalePrice",
)

### Résidus en fonction des variables quantitatives


In [21]:
numerical_cols = [
    col
    for col in df_test_stack.columns
    if pd.api.types.is_any_real_numeric_dtype(df_test_stack[col])
]

scat(
    df_test_stack,
    "LotFrontage",
    "residus",
    numerical_cols,
)

### Résidus en fonction des variables qualitatives sélectionnées


In [22]:
categorical_cols = [
    col for col in df_test_stack.columns if df_test_stack[col].dtype == "object"
]

violin(df_test_stack, "MSSubClass", "residus", categorical_cols)

C'est pire pour l'instant. Donc il semblerait que ce ne soit pas une bonne piste. Étant donné qu'il n'y a pas de méthode stepwise codé nativement en python, je vais créer la fonction et tenter de l'appliquer à la fois sur le modèle unique, et sur les modèles segmentés.

## Stepwise fait maison


Ancienne version

```python
def stepwise_selection(
    train: pd.DataFrame,
    test: pd.DataFrame,
    target: str,
    metric_name: str = "mae",
    initial_list: list = [],
    verbose: bool = True,
) -> list:
    """
    Perform stepwise selection to identify the optimal set of features for a GLM using a special metric.

    Parameters:
    - train: training dataframe
    - test: test dataframe (to evaluate performance)
    - target: target of the dataframe
    - metric_name: the metric to evaluate
    - initial_list: List of initial features to include, except the target
    - verbose: Whether to print the process

    Returns:
    - List of selected features
    """

    # Test avant tout chose que la target n'est pas dans les variables explicatives
    if target in initial_list:
        raise ValueError("La variable à expliquer ne peut pas être explicative ici!")

    # Dictionnaire des metriques disponibles
    metrics = {
        "mae": mean_absolute_error,
        "rmse": root_mean_squared_error,
        "mee": median_absolute_error,
        "me": max_error,
        "r2": r2_score,
        "bic": "bic",
    }

    # Tests verifiant si la metrique selectionnée est disponible
    if metric_name not in metrics:
        raise ValueError(
            f"Metric '{metric_name}' is not supported. Choose from {list(metrics.keys())}."
        )

    # Initialisations
    # Metrique
    metric = metrics[metric_name]
    # Variables explicatives initiales
    included = initial_list
    # Liste des variables categorielles parmis le jeux d'entrainement
    categorical_cols = [col for col in train.columns if train[col].dtype == "object"]

    # Ignore un message d'erreur pour le cas de figure loi gamma et famille identique
    warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

    # Boucle
    while True:
        # Init
        changed = False
        best_feature = None
        worst_feature = None

        # Formule actuelle
        formula = f"{target} ~ {' + '.join([f'C({col})' if col in categorical_cols else col for col in included])}"

        # Modele actuel
        model = smf.glm(
            formula=formula,
            data=train,
            family=sm.families.Gamma(link=sm.families.links.Identity()),
        ).fit()

        # Metrique actuelle
        if metric_name == "bic":
            current_metric = model.bic_llf
        else:
            current_metric = metric(test[target], model.predict(test[included]))

        # Test de rajout d'une variable (forward step)
        # Liste des variables a tester
        excluded = list(set(train.columns) - set(included + [target]))

        # Parcours les variables a tester
        for new_col in excluded:
            # Formule testée
            formula = f"{target} ~ {' + '.join([f'C({col})' if col in categorical_cols else col for col in included + [new_col]])}"

            # Modele testé
            model = smf.glm(
                formula=formula,
                data=train,
                family=sm.families.Gamma(link=sm.families.links.Identity()),
            ).fit()

            # Nouvelle metrique suite au rajout d'une variable
            if metric_name == "bic":
                new_metric = model.bic_llf
            else:
                new_metric = metric(
                    test[target], model.predict(test[included + [new_col]])
                )

            # Calcul de la nouvelle metric et rajout ou non de la variable explicative en fonction
            if metric_name == "r2":
                if new_metric > current_metric:
                    current_metric = new_metric
                    best_feature = new_col
            elif new_metric < current_metric:
                current_metric = new_metric
                best_feature = new_col

        # S'il y a une variable qui vaut le coup, rajout dans la selection
        if best_feature is not None:
            included.append(best_feature)
            changed = True
            if verbose:
                print(f"Ajout de {best_feature} avec {metric_name} : {current_metric}")

        # Test de supperssion d'une variable (Backward step)
        for inc_col in included:
            # Liste temporaire des variables selectionné sauf une
            temp_included = included.copy()
            temp_included.remove(inc_col)

            # Nouvelle formule testée
            formula = f"{target} ~ {' + '.join([f'C({col})' if col in categorical_cols else col for col in temp_included])}"

            # Nouveau modèle testé
            model = smf.glm(
                formula=formula,
                data=train,
                family=sm.families.Gamma(link=sm.families.links.Identity()),
            ).fit()

            # Nouvelle metrique suite a la suppression d'une variable
            if metric_name == "bic":
                new_metric = model.bic_llf
            else:
                new_metric = metric(test[target], model.predict(test[temp_included]))

            # Calcul de la nouvelle metric et suppression ou non de la variable explicative en fonction
            if metric_name == "r2":
                if new_metric > current_metric:
                    current_metric = new_metric
                    worst_feature = inc_col
            elif new_metric < current_metric:
                current_metric = new_metric
                worst_feature = inc_col

        # S'il y a une variable qui ne vaut pas le coup, suppression dans la selection
        if worst_feature is not None:
            included.remove(worst_feature)
            changed = True
            if verbose:
                print(
                    f"Suppression de {worst_feature} avec {metric_name} : {current_metric}"
                )

        if not changed:
            break

    return included

selection_stepwise_pt1 = stepwise_selection(
    df_train_pt1, df_test_pt1, "SalePrice", "bic", selection
)
```


### Fonction pour évaluer les performances du modèle


In [35]:
def evaluate_model(formula, train_data, test_data, target, metric, feature=None):
    """
    Evaluate a model with the given formula and return the metric value.

    Parameters:
    - formula: The model formula.
    - train_data: The training data.
    - test_data: The test data.
    - target: The target variable name.
    - metric: The metric to evaluate.
    - feature: Optional. The feature being evaluated. Default is None.

    Returns:
    - A tuple containing the feature (if provided) and the metric value.
    """
    try:
        model = smf.glm(
            formula=formula,
            data=train_data,
            family=sm.families.Gamma(link=sm.families.links.Identity()),
        ).fit()
        if metric == "bic":
            return (feature, model.bic_llf) if feature is not None else model.bic_llf
        else:
            predictions = model.predict(test_data)
            metric_value = metric(test_data[target], predictions)
            return (feature, metric_value) if feature is not None else metric_value
    except Exception as e:
        print(f"Error fitting model with formula {formula}: {e}")
        if metric == "bic":
            return (feature, float("inf")) if feature is not None else float("inf")
        else:
            return (feature, float("-inf")) if feature is not None else float("-inf")


# Example usage
# result = evaluate_model(formula, train_data, test_data, target, metric)
# feature_result = evaluate_model(formula, train_data, test_data, target, metric, feature="example_feature")

### Fonction stepwise


In [45]:
def stepwise_selection(
    train: pd.DataFrame,
    test: pd.DataFrame,
    target: str,
    metric_name: str = "mae",
    initial_list: list = [],
    verbose: bool = True,
) -> list:
    """
    Perform stepwise selection to identify the optimal set of features for a GLM using a special metric.

    Parameters:
    - train: training dataframe
    - test: test dataframe (to evaluate performance)
    - target: target of the dataframe
    - metric_name: the metric to evaluate
    - initial_list: List of initial features to include, except the target
    - verbose: Whether to print the process

    Returns:
    - List of selected features
    """

    if target in initial_list:
        raise ValueError("La variable à expliquer ne peut pas être explicative ici!")

    metrics = {
        "mae": mean_absolute_error,
        "rmse": root_mean_squared_error,
        "mee": median_absolute_error,
        "me": max_error,
        "r2": r2_score,
        "bic": "bic",
    }

    if metric_name not in metrics:
        raise ValueError(
            f"Metric '{metric_name}' is not supported. Choose from {list(metrics.keys())}."
        )

    metric = metrics[metric_name]
    included = initial_list.copy()
    categorical_cols = [col for col in train.columns if train[col].dtype == "object"]

    warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

    while True:
        changed = False
        best_feature = None
        worst_feature = None
        forward_results = {}
        backward_results = {}

        # Init formula
        formula = f"{target} ~ {' + '.join([f'C({col})' if col in categorical_cols else col for col in included])}"

        # Init metric
        current_metric = evaluate_model(formula, train, test, target, metric)

        # Forward step
        excluded = list(set(train.columns) - set(included + [target]))
        forward_results = dict(
            Parallel(n_jobs=-1)(
                delayed(evaluate_model)(
                    f"{target} ~ {' + '.join([f'C({col})' if col in categorical_cols else col for col in included + [new_col]])}",
                    train,
                    test,
                    target,
                    metric,
                    new_col,
                )
                for new_col in excluded
            )
        )

        if metric_name == "r2":
            best_feature = max(forward_results, key=forward_results.get, default=None)
        else:
            best_feature = min(forward_results, key=forward_results.get, default=None)

        if best_feature is not None and forward_results[best_feature] != current_metric:
            included.append(best_feature)
            changed = True
            if verbose:
                print(
                    f"Ajout de {best_feature} avec {metric_name} : {forward_results[best_feature]}"
                )

        # Backward step
        backward_results = dict(
            Parallel(n_jobs=-1)(
                delayed(evaluate_model)(
                    f"{target} ~ {' + '.join([f'C({col})' if col in categorical_cols else col for col in included if col != inc_col])}",
                    train,
                    test,
                    target,
                    metric,
                    inc_col,
                )
                for inc_col in included
            )
        )

        if metric_name == "r2":
            worst_feature = max(
                backward_results, key=backward_results.get, default=None
            )
        else:
            worst_feature = min(
                backward_results, key=backward_results.get, default=None
            )

        if (
            worst_feature is not None
            and backward_results[worst_feature] != current_metric
        ):
            included.remove(worst_feature)
            changed = True
            if verbose:
                print(
                    f"Suppression de {worst_feature} avec {metric_name} : {backward_results[worst_feature]}"
                )

        if not changed or worst_feature == best_feature:
            break

    return included


# Example usage
# selected_features = stepwise_selection(train_df, test_df, "SalePrice", metric_name='mae', initial_list=selection)

### Application de la fonction

#### Quelques modalités trop marginales qui ont posées problème


In [ ]:
for df in [train, test]:
    df.rename(columns={"3SsnPorch": "TroisSsnPorch"}, inplace=True)

train_pt1 = train[train["SecondFlrSF"] < 650].copy()
train_pt2 = train[train["SecondFlrSF"] >= 650].copy()

# Suppression de certaines colonnes, avec des modalité marginales mais ayant une version equivalente
# Foundation est enleve mais je ne pense pas qu'elle ai un reel impact. Reaggreger sinon
df_train_pt1, df_test_pt1 = train_test_split(
    train_pt1.drop(
        columns=[
            "Alley",
            "ExterCond",
            "ExterCond_ord",
            "Heating",
            "HeatingQC",
            "HeatingQC_ord",
            "Foundation",
            "Exterior1st",
            "Exterior2nd",
            "RoofStyle",
            "SaleType",
        ]
    ),
    test_size=0.2,
    random_state=42,
    stratify=train_pt1["MSZoning"],
)

df_train_pt2, df_test_pt2 = train_test_split(
    train_pt2.drop(
        columns=[
            "Alley",
            "ExterCond",
            "ExterCond_ord",
            "Heating",
            "HeatingQC",
            "HeatingQC_ord",
            "Foundation",
            "Exterior1st",
            "Exterior2nd",
            "RoofStyle",
            "SaleType",
            "RoofMatl",
            "Condition1",
            "PoolQC",
            "PoolQC_ord",
        ]
    ),
    test_size=0.2,
    random_state=42,
    stratify=train_pt2["MSZoning"],
)

#### Application pour le premier segment


In [82]:
selection_stepwise_pt1 = stepwise_selection(
    df_train_pt1, df_test_pt1, "SalePrice", metric_name="rmse", initial_list=selection
)

Ajout de OverallQual avec rmse : 23323.890786262018
Suppression de BsmtExposure_ord avec rmse : 23199.49204752648
Ajout de OverallCond avec rmse : 22500.59771815655
Suppression de OverallQual_ord avec rmse : 22373.750831607045
Ajout de SaleCondition_optb avec rmse : 21911.947398029002
Suppression de BsmtQual avec rmse : 21733.604203455812
Ajout de WoodDeckSF avec rmse : 21438.208883347335
Suppression de Fireplaces_optb avec rmse : 21260.729920338377
Ajout de BsmtCond avec rmse : 20998.614843626532
Suppression de GarageQual_agg avec rmse : 20935.7773452953
Ajout de MasVnrType avec rmse : 20762.940710057417
Suppression de CentralAir avec rmse : 20770.17092167269
Ajout de Condition2 avec rmse : 20627.471937414135
Suppression de HalfBath_tot avec rmse : 20659.903605321975
Ajout de BsmtFinSF1 avec rmse : 20471.456010820508
Suppression de BsmtFinType1_ord avec rmse : 20504.990395054287
Ajout de KitchenAbvGr avec rmse : 20370.347187825704
Suppression de LotFrontage avec rmse : 20455.650608170

#### Application pour le deuxième segment


In [92]:
selection_stepwise_pt2 = stepwise_selection(
    df_train_pt2, df_test_pt2, "SalePrice", metric_name="rmse", initial_list=selection
)

Ajout de BldgType avec rmse : 63096.687298764395
Suppression de TotalBsmtSF avec rmse : 54112.7180069012
Ajout de BsmtUnfSF avec rmse : 53185.30307359685
Suppression de FirstFlrSF avec rmse : 46345.923145924775
Ajout de LotShape avec rmse : 45489.820738039656
Suppression de GarageQual_agg avec rmse : 44352.8543364497
Ajout de OverallCond avec rmse : 43042.998038087244
Suppression de GarageArea avec rmse : 42161.91921661608
Ajout de FireplaceQu_ord avec rmse : 41446.64091586685
Suppression de LotArea avec rmse : 40374.98595673706
Ajout de HouseStyle avec rmse : 39548.775631149285
Suppression de FullBath_tot avec rmse : 39248.9306212837
Ajout de FullBath avec rmse : 38030.547290700066
Suppression de Fireplaces_optb avec rmse : 37842.81650559577
Ajout de FullBath_optb avec rmse : 36147.78496877912
Suppression de BsmtFinType1_ord avec rmse : 36050.63224170667
Ajout de GarageCars avec rmse : 35481.278255382924
Suppression de BsmtUnfSF avec rmse : 35429.88343786963
Ajout de SaleCondition_opt

### Analyses des performances pour les modèles segmentés

#### Calculs des prédictions


In [93]:
formule = f"{debut_formule} + {' + '.join([f'C({col})' if col in categorical_cols else col for col in selection_stepwise_pt1])}"

res1 = smf.glm(
    formula=formule,
    data=df_train_pt1,
    family=sm.families.Gamma(link=sm.families.links.Identity()),
).fit()

df_test_pt1["SalePrice_pred_stack"] = res1.predict(df_test_pt1)

In [94]:
formule = f"{debut_formule} + {' + '.join([f'C({col})' if col in categorical_cols else col for col in selection_stepwise_pt2])}"

res2 = smf.glm(
    formula=formule,
    data=df_train_pt2,
    family=sm.families.Gamma(link=sm.families.links.Identity()),
).fit()

df_test_pt2["SalePrice_pred_stack"] = res2.predict(df_test_pt2)

In [95]:
df_test_stack = pd.concat([df_test_pt1, df_test_pt2])

#### Quelques métriques bien connues


In [96]:
plot_perf(df_test_stack["SalePrice"], df_test_stack["SalePrice_pred_stack"])

,MAE,RMSE,MEE,ME,R2
0,15614.581606,25204.942902,10397.065094,171325.615681,0.909726


#### Forme des résidus


In [97]:
df_test_stack["residus"] = (
    df_test_stack["SalePrice"] - df_test_stack["SalePrice_pred_stack"]
)
residuals_density(df_test_stack, "residus")

#### Résidus en fonction du Prix de vente


In [98]:
scat_res_price(
    df_test_stack,
    "residus",
    "SalePrice",
)

#### Résidus en fonction des variables quantitatives


In [99]:
numerical_cols = [
    col
    for col in df_test_stack.columns
    if pd.api.types.is_any_real_numeric_dtype(df_test_stack[col])
]

scat(
    df_test_stack,
    "LotFrontage",
    "residus",
    numerical_cols,
)

#### Résidus en fonction des variables qualitatives sélectionnée


In [100]:
categorical_cols = [
    col for col in df_test_stack.columns if df_test_stack[col].dtype == "object"
]

violin(df_test_stack, "MSSubClass", "residus", categorical_cols)

### Modèle complet


In [ ]:
df_train, df_test = train_test_split(
    train.drop(
        columns=[
            "Alley",
            "Condition2",
            "RoofMatl",
            "Exterior1st",
            "Exterior2nd",
            "Electrical",
        ]
    ),
    test_size=0.3,
    random_state=42,
    stratify=train["MSZoning"],
)

In [110]:
selection_stepwise_complet = stepwise_selection(
    df_train, df_test, "SalePrice", metric_name="rmse", initial_list=selection
)

Ajout de MSSubClass avec rmse : 36919.019188708095
Suppression de TotalBsmtSF avec rmse : 34804.977350958856
Ajout de GarageCars avec rmse : 34342.859525019296
Suppression de FirstFlrSF avec rmse : 32807.08826523926
Ajout de TotRmsAbvGrd avec rmse : 31980.919376984708
Suppression de LotArea avec rmse : 31291.710294854547
Ajout de OverallCond avec rmse : 30745.174634064017
Suppression de GarageArea avec rmse : 30401.11324643578
Ajout de FireplaceQu avec rmse : 30253.857870608957
Suppression de LotFrontage avec rmse : 30024.66874331799
Ajout de Neighborhood avec rmse : 29880.606065432436
Suppression de BsmtQual avec rmse : 29643.70890211501
Ajout de LandContour avec rmse : 29497.431607110935
Suppression de CentralAir avec rmse : 29475.83217878794
Ajout de Exterior2nd_agg avec rmse : 29347.13864786884
Suppression de Neighborhood_agg2 avec rmse : 29347.049785247313
Ajout de Condition1 avec rmse : 29231.543990280115
Suppression de Fireplaces_optb avec rmse : 29251.62202473345
Ajout de HalfB

### Analyses des performances pour le modèle complet

#### Calculs des prédictions


In [111]:
formule = f"{debut_formule} + {' + '.join([f'C({col})' if col in categorical_cols else col for col in selection_stepwise_complet])}"

res0 = smf.glm(
    formula=formule,
    data=df_train,
    family=sm.families.Gamma(link=sm.families.links.Identity()),
).fit()

df_test["SalePrice_pred"] = res0.predict(df_test)

#### Quelques métriques bien connues


In [112]:
plot_perf(df_test["SalePrice"], df_test["SalePrice_pred"])

,MAE,RMSE,MEE,ME,R2
0,18231.127594,29072.213192,12270.74138,217226.54934,0.835364


#### Forme des résidus


In [ ]:
df_test["residus"] = df_test["SalePrice"] - df_test["SalePrice_pred"]
residuals_density(df_test, "residus")

#### Résidus en fonction du Prix de vente


In [115]:
scat_res_price(
    df_test,
    "residus",
    "SalePrice",
)

#### Résidus en fonction des variables quantitatives


In [116]:
numerical_cols = [
    col
    for col in df_test.columns
    if pd.api.types.is_any_real_numeric_dtype(df_test[col])
]

scat(
    df_test,
    "LotFrontage",
    "residus",
    numerical_cols,
)

#### Résidus en fonction des variables qualitatives sélectionnée


In [ ]:
categorical_cols = [col for col in df_test.columns if df_test[col].dtype == "object"]

violin(df_test, "MSSubClass", "residus", categorical_cols)

## Remarques globales

En fin de compte, la segmentation s'est avérée être une bonne idée, car elle a amélioré les performances. Souvent, lors de la sélection pas à pas, certaines variables sont remplacées par d'autres contenant des informations similaires. Les sélections pour les modèles segmentés ne sont pas identiques, ce qui démontre l'intérêt de ce découpage. Cependant, des erreurs importantes persistent dans certains cas. Les résidus restent comparables en termes de forme et de valeurs. D'autres découpages pourraient être envisagés. Une validation croisée pourrait également être réalisée, voire un stacking, mais il semble opportun de passer à des méthodes ensemblistes.
